## 最小编辑距离（Minimum Edit Distance）

* **1. 核心底层逻辑：最小编辑距离 (Minimum Edit Distance)**
  
    这是教电脑判断“两个词有多像”的数学基础。

    * **大白话定义：** 把单词 A 变成单词 B，最少需要动几次手术（几步操作）。

    * **允许的四种“手术”：** 插入字符 (Insert)、删除字符 (Delete)、替换字符 (Substitute)、调换相邻字符 (Transpose)。

    * **经典案例：** 从 intention 变成 execution，最小编辑距离是 5（视频原话：删 I，加 C，替换第一处的 N为E、T为X、第二处的 N为U）。

    * **铁律：** 距离越小，两个词越像（距离为 0 代表一模一样）。
  
* **2. 实战大杀器：thefuzz 相似度计算库**
  
        知道了底层逻辑，现实中我们不需要自己手写算法去算距离，直接调用 Python 的 thefuzz 包（以前叫 fuzzywuzzy）就行了。

    * **对比两个词 (fuzz.WRatio)：**

      * 传入两个字符串，它会给你返回一个 0 到 100 的得分。0 代表毫不相干，100 代表完全匹配。

    * **⚠️ 注意避坑：** 
  
      *  这个分数跟刚才的“距离”是反着的！这里是分数越高，两个词越像。

    * **鲁棒性极高：**
  
      * 如果拿 Houston Rockets（休斯顿火箭队）和 Rockets（火箭队）去比，或者拿顺序颠倒的句子去比，传统的等于号 == 会直接判错，但 WRatio 依然能给出极高的相似度分数。

    * **示例代码：**

In [ ]:
# % 符号是 Jupyter 的魔法命令，它能确保这个库准确无误地装进当前 Notebook 正在使用的那个虚拟环境里。
# 后面的 [speedup] 是个进阶小技巧，它会顺带安装一个叫 python-Levenshtein 的底层 C 语言加速包，能让电脑算相似度的速度直接起飞！
%pip install thefuzz[speedup]

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 9.1 MB/s  0:00:00

   ---------------------------------------- 2/2 [thefuzz]

Note: you may need to restart the kernel to use updated packages.


In [4]:
 # 导入 fuzz 模块
from thefuzz import fuzz

print("--- 1. 基础拼写错误 (Typo) ---")
# 传统的 == 绝对会报错，但 WRatio 能看出它们只是拼错了一个字母
score1 = fuzz.WRatio("reading", "rading")
print(f"reading vs rading: {score1} 分\n")

print("--- 2. 长短不一的包含关系 (Partial Match) ---")
# 休斯顿火箭队 vs 火箭队。WRatio 非常聪明，它发现 "Rockets" 被完全包含在长字符串里。
score2 = fuzz.WRatio("Houston Rockets", "Rockets")
print(f"Houston Rockets vs Rockets: {score2} 分\n")

print("--- 3. 顺序完全颠倒 (Different Orderings) ---")
# 湖人打火箭 vs 火箭打湖人。不仅缺字，顺序还全乱了。
string_a = "Houston Rockets vs Los Angeles Lakers"
string_b = "Lakers vs Rockets"
score3 = fuzz.WRatio(string_a, string_b)
print(f"长短不一且乱序比对: {score3} 分\n")

print("--- 4. 传统等于号 (==) 的无情对比 ---")
print(f"传统 == 判断 (Houston Rockets == Rockets): {'Houston Rockets' == 'Rockets'}")

--- 1. 基础拼写错误 (Typo) ---
reading vs rading: 92 分

--- 2. 长短不一的包含关系 (Partial Match) ---
Houston Rockets vs Rockets: 90 分

--- 3. 顺序完全颠倒 (Different Orderings) ---
长短不一且乱序比对: 86 分

--- 4. 传统等于号 (==) 的无情对比 ---
传统 == 判断 (Houston Rockets == Rockets): False


* **3. 进阶神技：从茫茫人海中找兄弟 (对比数组)**
  
    * 如果我手里有一个错别字，想去一个包含 100 个正确单词的词库里找“最可能是谁”，总不能手写 100 次 WRatio 吧？

    * `使用工具：``process.extract`(目标词, 词库数组, limit=返回数量)

    * `返回值：`它会返回一个列表，里面装着它为你挑选出的候选人。每个候选人包含三个信息：(匹配到的词, 相似度得分, 在原数组里的索引值)。

    * `示例代码：`

In [8]:
import pandas as pd
from thefuzz import process

# 1. 假设这是干净的、标准的“州名词库” (用 Pandas Series 装着)
correct_states = pd.Series([
    "California", 
    "New York", 
    "Texas", 
    "Florida", 
    "Washington"
])

# 2. 这是调查问卷里用户乱填的一个错别字
typo_word = "New Yrok"  # 故意把 r 和 o 写反

print(f"🔍 正在为脏数据 '{typo_word}' 寻找最匹配的亲兄弟...\n")

# 3. 召唤 process.extract，在 correct_states 里找最像的 2 个候选人
# 参数：(目标词, 词库, limit=返回几个候选人)
matches = process.extract(typo_word, correct_states, limit=2)
print(f"打印'matches'的输出结果(一个包含元组的列表):") 
print(matches)
print("--- 🏆 匹配结果排行榜 ---")
for match in matches:
    print(match)

🔍 正在为脏数据 'New Yrok' 寻找最匹配的亲兄弟...

打印'matches'的输出结果(一个包含元组的列表):
[('New York', 88, 1), ('Texas', 26, 2)]
--- 🏆 匹配结果排行榜 ---
('New York', 88, 1)
('Texas', 26, 2)


* **4. 终极应用套路：用相似度批量清洗脏数据 (Collapsing Categories)**

    * 这是这节课最核心的代码逻辑。当一列数据里充斥着成百上千种错别字（比如州名、城市名拼错），手动替换是不可能完成的任务。

    * **自动化清洗的四步法：**

    * **遍历标准库：** 拿出一个正确的州名（比如 New York）。

    * **大撒网捞鱼：** 用 process.extract() 去全是错别字的调查问卷表里，把所有跟 New York 长得像的词都揪出来。

    * **划定及格线：** 用一个 if 语句卡住分数，比如只保留相似度 >= 80 分的那些词（分数太低可能是真的另一个州，不能误杀）。

    * **一键替换：** 用 Pandas 的 .loc 方法，把揪出来的高分错别字，统统替换成标准的 New York。
  
    * **示例代码：**

In [9]:
import pandas as pd
from thefuzz import process

# --- 准备工作：制造案发现场 ---
# 1. 我们的脏数据表 (里面全是乱填的州名)
survey = pd.DataFrame({
    'respondent_id': [1, 2, 3, 4, 5, 6],
    'state': ['California', 'Cali-fornia', 'New York', 'New Yrok', 'NYork', 'Texas']
})

# 2. 干净的标准字典库
correct_states = pd.Series(['California', 'New York', 'Texas'])

print("❌ 清洗前的脏数据：")
print(survey)
print("-" * 40)

# ==========================================
# 🚀 核心逻辑：自动化清洗四步法
# ==========================================

# 第一步：遍历标准库 (挨个拿出正确的州名)
for state in correct_states:
    
    # 第二步：大撒网捞鱼 
    # 在 survey['state'] 列里找跟当前 correct_state 相似的词。
    # ⚠️ 关键点：为了不错过任何一个错别字，要把 limit 设为 survey 表的总行数！
    matches = process.extract(state, survey['state'], limit=survey.shape[0])
    
    # matches 返回的是一个列表，里面长这样：[('New York', 100, 2), ('New Yrok', 95, 3), ...]
    for potential_match in matches:
        match_word = potential_match[0]  # 提取抓到的词 (比如 'New Yrok')
        score = potential_match[1]       # 提取相似度得分 (比如 95)
        
        # 第三步：划定及格线
        if score >= 80:
            
            # 第四步：一键替换
            # 用 .loc 定位到 survey 表中所有 'state' 列等于 match_word 的行，把它们强制改成标准的 state
            survey.loc[survey['state'] == match_word, 'state'] = state

# ==========================================

print("✨ 见证奇迹：清洗后的干净数据：")
print(survey)

❌ 清洗前的脏数据：
   respondent_id        state
0              1   California
1              2  Cali-fornia
2              3     New York
3              4     New Yrok
4              5        NYork
5              6        Texas
----------------------------------------
✨ 见证奇迹：清洗后的干净数据：
   respondent_id       state
0              1  California
1              2  California
2              3    New York
3              4    New York
4              5    New York
5              6       Texas


### if条件判断的分数取值(阈值)
* **1. 先用“黄金法则” 80 分作为起点**

    经过无数程序员的血泪测试，对于普通的英文单词和短语，80 分是一个性价比最高的阈值。它能挡住大部分张冠李戴的词，又能放进大部分因为手抖打错的字。

* **2. 抽查“边缘地带”（最重要的一步！）**

    代码跑完之后，绝对不能直接把结果交差。把得分在 75 到 85 之间的那些匹配对打印出来看一眼。

    如果看到 score=82 的匹配是 McDonalds 和 MacDonald's，说明 80 分定得很完美。

    如果看到 score=81 的匹配把 Apple 和 Apply 连在一起了，就知道 80 分定低了，得调到 85。

* **3. 根据“业务容忍度”倾斜**

    最后，要问问自己：这个项目，是漏掉更可怕，还是认错更可怕？

    宁可错杀，不可放过（降低及格线，比如 70 分）：如果任务是把所有可能的重复客户都找出来进行人工审核，那宁愿让系统多找点（哪怕找错），也不能漏掉。

    宁可漏掉，不可认错（提高及格线，比如 90 分）：如果这是用来合并两个银行账户的代码，只要稍微有一点不像，就坚决不能合并。合并错了，钱就打给别人了，这时候你就算设 95 分都不为过。

    * **阈值调优实战代码:**

In [10]:
import pandas as pd
from thefuzz import process

# --- 准备工作：制造包含各种边缘情况的脏数据 ---
correct_brands = pd.Series(['Apple', 'McDonalds', 'Microsoft'])
survey = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6],
    'brand_name': [
        'Apple Inc.',   # 正常变体
        'Apply',        # 极度危险的易混淆词 (不同公司，但字母极像)
        "MacDonald's",  # 典型的拼写/符号错误
        'mcdonalds',    # 大小写错误
        'Micro soft',   # 空格错误
        'Sony'          # 完全无关
    ]
})

print("🚨 第一步：进入【边缘审查模式】(不修改数据，只打印)")
print("-" * 50)
# 专门抓取 70 到 85 分之间的匹配项，打印出来让人工过目
for standard_word in correct_brands:
    matches = process.extract(standard_word, survey['brand_name'], limit=survey.shape[0])
    for match_word, score, index in matches:
        # 截获边缘地带的分数！
        if 70 <= score <= 85:
            print(f"⚠️ 审查目标：标准词 [{standard_word:10}] <--> 疑似词 [{match_word:12}] | 得分: {score}")

print("\n" + "=" * 50 + "\n")

# =======================================================
# 🧠 第二步：大脑决策期 (根据上面的打印结果，设定最终阈值)
# 假设我们看到 Apple 和 Apply 的得分是 81 分。
# 业务要求：绝对不能把 Apply(另一家公司) 算成 Apple！
# 决策：80分及格线太低了，必须提高到 85 分或 90 分！
# =======================================================

FINAL_THRESHOLD = 90  # 我们采用“宁可漏掉，不可认错”的严格策略

print(f"🚀 第二步：进入【最终清洗模式】(当前阈值: {FINAL_THRESHOLD})")
print("-" * 50)

# 创建一个清洗后的副本，防止把原数据搞坏
cleaned_survey = survey.copy()

for standard_word in correct_brands:
    matches = process.extract(standard_word, cleaned_survey['brand_name'], limit=cleaned_survey.shape[0])
    for match_word, score, index in matches:
        if score >= FINAL_THRESHOLD:
            # 执行替换
            cleaned_survey.loc[cleaned_survey['brand_name'] == match_word, 'brand_name'] = standard_word
            
print("✨ 清洗完成！最终结果：")
print(cleaned_survey)

🚨 第一步：进入【边缘审查模式】(不修改数据，只打印)
--------------------------------------------------
⚠️ 审查目标：标准词 [Apple     ] <--> 疑似词 [Apply       ] | 得分: 80


🚀 第二步：进入【最终清洗模式】(当前阈值: 90)
--------------------------------------------------
✨ 清洗完成！最终结果：
   id brand_name
0   1      Apple
1   2      Apply
2   3  McDonalds
3   4  McDonalds
4   5  Microsoft
5   6       Sony
